In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [3]:
df = pd.read_csv("../data/keystroke_data.csv")

df = df.sort_values(by=["PARTICIPANT_ID", "PRESS_TIME"])

In [4]:
df["hold_time"] = df["RELEASE_TIME"] - df["PRESS_TIME"]

df["next_press"] = df.groupby("PARTICIPANT_ID")["PRESS_TIME"].shift(-1)
df["flight_time"] = df["next_press"] - df["RELEASE_TIME"]

df = df.dropna()
df = df[(df["hold_time"] > 0) & (df["flight_time"] > 0)]

In [5]:
top_users = df["PARTICIPANT_ID"].value_counts().head(50).index
df = df[df["PARTICIPANT_ID"].isin(top_users)]

In [6]:
sequence_length = 30

sequences = []
labels = []

for user in df["PARTICIPANT_ID"].unique():
    user_df = df[df["PARTICIPANT_ID"] == user]
    data = user_df[["hold_time", "flight_time"]].values

    for i in range(len(data) - sequence_length):
        sequences.append(data[i:i+sequence_length])
        labels.append(user)

X = np.array(sequences)
y = np.array(labels)

print(X.shape, y.shape)

(32336, 30, 2) (32336,)


In [7]:
num_samples, seq_len, num_features = X.shape

X_reshaped = X.reshape(-1, num_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)

X = X_scaled.reshape(num_samples, seq_len, num_features)

In [8]:
le = LabelEncoder()
y = le.fit_transform(y)

num_classes = len(set(y))
print("Classes:", num_classes)

Classes: 50


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [10]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [11]:
class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model, num_heads, num_layers, num_classes):
        super().__init__()
        
        self.input_fc = nn.Linear(input_size, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        x = self.input_fc(x)
        x = self.transformer(x)
        x = x[:, -1, :]  # last token
        x = self.fc(x)
        return x

In [12]:
model = TransformerModel(
    input_size=2,
    d_model=64,
    num_heads=4,
    num_layers=2,
    num_classes=num_classes
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
batch_size = 64
epochs = 20

for epoch in range(epochs):
    model.train()
    
    perm = torch.randperm(X_train.size(0))
    total_loss = 0
    
    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]
        
        batch_X = X_train[idx]
        batch_y = y_train[idx]
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 974.5456
Epoch 2, Loss: 653.7739
Epoch 3, Loss: 538.4394
Epoch 4, Loss: 435.0730
Epoch 5, Loss: 370.1366
Epoch 6, Loss: 322.0645
Epoch 7, Loss: 285.4369
Epoch 8, Loss: 262.3064
Epoch 9, Loss: 236.1492
Epoch 10, Loss: 210.3259
Epoch 11, Loss: 191.5890
Epoch 12, Loss: 172.7900
Epoch 13, Loss: 159.9033
Epoch 14, Loss: 147.7020
Epoch 15, Loss: 147.4623
Epoch 16, Loss: 131.8478
Epoch 17, Loss: 124.4403
Epoch 18, Loss: 120.8628
Epoch 19, Loss: 108.6468
Epoch 20, Loss: 105.7050


In [14]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    _, pred = torch.max(outputs, 1)
    
    acc = (pred == y_test).float().mean()

print(" Transformer Accuracy:", acc.item())

 Transformer Accuracy: 0.9630488753318787


In [15]:
model.eval()

with torch.no_grad():
    train_pred = model(X_train).argmax(1)
    train_acc = (train_pred == y_train).float().mean()

    test_pred = model(X_test).argmax(1)
    test_acc = (test_pred == y_test).float().mean()

print("Train Accuracy:", train_acc.item())
print("Test Accuracy:", test_acc.item())

Train Accuracy: 0.9785062670707703
Test Accuracy: 0.9630488753318787


In [16]:
torch.save(model.state_dict(), "../models/transformer_model.pth")

In [17]:
from sklearn.metrics import top_k_accuracy_score

model.eval()

with torch.no_grad():
    outputs = model(X_test)
    probs = torch.softmax(outputs, dim=1).numpy()

top3 = top_k_accuracy_score(y_test.numpy(), probs, k=3)

print(" Transformer Top-3 Accuracy:", top3)

 Transformer Top-3 Accuracy: 0.9979901051329623


In [19]:
def get_embedding(model, x):
    with torch.no_grad():
        x = model.input_fc(x)
        x = model.transformer(x)
        x = x[:, -1, :]   # last token
    return x

# Get embeddings for all users
embeddings = []
labels_list = []

for i in range(len(X)):
    x_tensor = torch.tensor(X[i], dtype=torch.float32).unsqueeze(0)
    
    emb = get_embedding(model, x_tensor)
    
    embeddings.append(emb.numpy()[0])
    labels_list.append(y[i])

embeddings = np.array(embeddings)
labels_list = np.array(labels_list)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (32336, 64)


In [20]:
np.save("../models/embeddings.npy", embeddings)
np.save("../models/labels.npy", labels_list)